## Gemini 3.1 Flash Lite — Structured Output Load Test

**Endpoint**: `databricks-gemini-3-1-flash-lite`

**Rate limits** ([docs](https://docs.databricks.com/aws/en/machine-learning/foundation-model-apis/limits#rate-limits-by-model)):
| Metric | Limit |
| --- | --- |
| Output Tokens Per Minute (OTPM) | 20,000 |
| Input Tokens Per Minute (ITPM) | 200,000 |
| Queries Per Hour (QPH) | 360,000 |

Limits are enforced via a **token bucket with burst buffer**, **sliding window**, **pre-admission checks**, and **credit-back** — see [How limits are tracked and enforced](https://docs.databricks.com/aws/en/machine-learning/foundation-model-apis/limits#how-limits-are-tracked-and-enforced) for details.

This notebook:
1. Sends **structured output** requests (`response_format: json_schema`) to extract product review analysis as typed JSON.
2. Runs a **concurrent load test** to probe the OTPM ceiling.
3. Applies **exponential backoff with jitter** on `429 REQUEST_LIMIT_EXCEEDED` errors so requests recover gracefully.

In [0]:
import os, time, json, random
import pandas as pd
from openai import OpenAI

# --- Databricks FMAPI client ---
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
if not workspace_url.startswith("https://"):
    workspace_url = f"https://{workspace_url}"

client = OpenAI(
    api_key=dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get(),
    base_url=f"{workspace_url}/serving-endpoints",
)
ENDPOINT = "databricks-gemini-3-1-flash-lite"

# --- Sample product reviews (in-memory) ---
REVIEWS = [
    ("Laptop",       "Great performance for the price. Battery lasts about 6 hours. Keyboard feels mushy but overall solid for a developer on a budget."),
    ("Headphones",   "Noise cancellation is decent. Sound quality is warm with good bass. Comfortable for long sessions. Bluetooth range could be better."),
    ("Standing Desk","Easy to assemble, sturdy. Electric motor is smooth and quiet. Wobbles slightly at max height."),
    ("Keyboard",     "Cherry MX Brown switches are tactile without being loud. RGB lighting is vivid. Aluminum frame, PBT keycaps feel great."),
    ("4K Monitor",   "Colors accurate out of the box. 144Hz refresh rate is smooth. HDR impressive for the price. Stand needs more adjustability."),
]

reviews_block = "\n".join(f"- {p}: {r}" for p, r in REVIEWS)

# --- Structured output JSON schema ---
RESPONSE_FORMAT = {
    "type": "json_schema",
    "json_schema": {
        "name": "product_review_analysis",
        "schema": {
            "type": "object",
            "properties": {
                "analyses": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "product":    {"type": "string"},
                            "sentiment":  {"type": "string", "enum": ["positive", "mixed", "negative"]},
                            "score":      {"type": "number"},
                            "strengths":  {"type": "array", "items": {"type": "string"}},
                            "weaknesses": {"type": "array", "items": {"type": "string"}},
                            "summary":    {"type": "string"},
                        },
                    },
                }
            },
        },
        "strict": True,
    },
}

PROMPT = f"""Analyze each product review. Return structured JSON.

Reviews:
{reviews_block}
"""

print(f"✅ Client ready — endpoint: {ENDPOINT}")
print(f"✅ {len(REVIEWS)} reviews loaded")
print(f"✅ JSON schema: product_review_analysis")

### Request helper — `call_structured()`

Wraps a single `chat.completions.create` call with `response_format: json_schema` and returns a flat result dict.

| Behaviour | Detail |
| --- | --- |
| **Exponential backoff + jitter** | On `429 REQUEST_LIMIT_EXCEEDED`, retries up to `max_retries` times with randomised delay (`base_delay × 2^attempt`, capped at `max_delay`). |
| **Empty-body retry** | Gemini occasionally returns an empty completion; this is treated as retryable. |
| **Cache busting** | When `bust_cache=True`, a unique nonce is appended to the prompt so the server can't short-circuit with a cached response. |

In [0]:
import uuid

def _extract_text(content):
    """Gemini may return content as a list of parts."""
    return content[0]["text"] if isinstance(content, list) and content else content

def _is_rate_limit(exc):
    msg = str(exc)
    return getattr(exc, "status_code", None) == 429 or "429" in msg or "REQUEST_LIMIT_EXCEEDED" in msg


def call_structured(req_id=0, *, max_tokens=4096, max_retries=5,
                    base_delay=2.0, max_delay=60.0, bust_cache=False):
    """Single structured-output request with exponential backoff on 429s."""
    prompt = PROMPT + f"\n<!-- req={req_id} nonce={uuid.uuid4().hex} -->" if bust_cache else PROMPT
    t0 = time.time()

    for attempt in range(1, max_retries + 2):
        t_call = time.time()
        try:
            resp = client.chat.completions.create(
                model=ENDPOINT,
                messages=[{"role": "user", "content": prompt}],
                response_format=RESPONSE_FORMAT,
                max_tokens=max_tokens, temperature=0.7,
            )
            raw = _extract_text(resp.choices[0].message.content)
            if not raw or not raw.strip():
                raise json.JSONDecodeError("Empty response body", "", 0)

            parsed = json.loads(raw)
            return dict(
                req_id=req_id, status="ok", attempts=attempt,
                prompt_tokens=resp.usage.prompt_tokens,
                completion_tokens=resp.usage.completion_tokens,
                finish_reason=resp.choices[0].finish_reason,
                products_returned=len(parsed.get("analyses", [])),
                elapsed=round(time.time() - t_call, 2),
                total_elapsed=round(time.time() - t0, 2), error=None,
            )

        except Exception as exc:
            if (_is_rate_limit(exc) or isinstance(exc, json.JSONDecodeError)) and attempt <= max_retries:
                delay = random.uniform(0, min(base_delay * 2 ** (attempt - 1), max_delay))
                print(f"  ⚠️  req={req_id} attempt {attempt}/{max_retries} — backoff {delay:.1f}s")
                time.sleep(delay)
                continue

            return dict(
                req_id=req_id, status="429_exhausted" if _is_rate_limit(exc) else "error",
                attempts=attempt, prompt_tokens=None, completion_tokens=None,
                finish_reason="error", products_returned=0,
                elapsed=round(time.time() - t_call, 2),
                total_elapsed=round(time.time() - t0, 2), error=str(exc)[:300],
            )

# Smoke test
result = call_structured(req_id=0)
print(f"✅ Smoke test: {result['status']} | {result['completion_tokens']} output tokens | "
      f"{result['products_returned']} products | {result['elapsed']}s")

### Concurrent load test

The next cell fires `TOTAL_REQUESTS` concurrent requests via a thread pool and measures:

* **Cumulative output-token throughput** against the documented **20 k OTPM** ceiling.
* **Retry distribution** — how many requests hit `429` and whether exponential backoff recovered them.

Tune `MAX_WORKERS` to increase or decrease pressure on the endpoint. The final chart overlays the observed token accumulation against the theoretical rate-limit line.

In [0]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.lines import Line2D

# --- Config ---
TOTAL_REQUESTS = 150
MAX_WORKERS    = 10
MAX_RETRIES    = 10
BASE_DELAY     = 2.0
MAX_DELAY      = 60.0
BUST_CACHE     = True

cache_label = "CACHE-BUST" if BUST_CACHE else "IDENTICAL"
print(f"🚀 {TOTAL_REQUESTS} reqs × {MAX_WORKERS} workers | backoff {BASE_DELAY}–{MAX_DELAY}s | {cache_label}\n")

# --- Fire requests ---
all_results, test_start = [], time.time()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    futs = {pool.submit(call_structured, i, max_retries=MAX_RETRIES,
                        base_delay=BASE_DELAY, max_delay=MAX_DELAY,
                        bust_cache=BUST_CACHE): i for i in range(TOTAL_REQUESTS)}
    for fut in as_completed(futs):
        r = fut.result()
        r["abs_end"] = time.time() - test_start
        all_results.append(r)
        tag = "✅" if r["error"] is None else "❌"
        retries = f"  (retries: {r['attempts']-1})" if r["attempts"] > 1 else ""
        print(f"  {tag} req={r['req_id']:>3}  tok={str(r['completion_tokens'] or 'N/A'):>5}  "
              f"t={r['abs_end']:.1f}s  {r['status']}{retries}")

wall = time.time() - test_start

# --- Summary ---
df = pd.DataFrame(all_results).sort_values("abs_end").reset_index(drop=True)
ok, fail = df[df["error"].isna()], df[df["error"].notna()]
total_otok = int(ok["completion_tokens"].sum()) if not ok.empty else 0
otpm = total_otok / wall * 60 if wall else 0

print(f"\n{'='*60}")
print(f"  {len(ok)}/{len(df)} succeeded | {len(df[df['attempts']>1])} retried | wall {wall:.0f}s")
if not ok.empty:
    print(f"  Avg: {ok['completion_tokens'].mean():.0f} tok/req, {ok['total_elapsed'].mean():.1f}s latency")
print(f"  Throughput: {total_otok:,} tokens → {otpm:,.0f} OTPM ({otpm/200:.1f}% of 20k limit)")
print(f"{'='*60}")

# --- Chart: cumulative tokens + retry scatter ---
sorted_ok = ok.sort_values("abs_end")
cum = sorted_ok["completion_tokens"].cumsum()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7), height_ratios=[3, 1], sharex=True)

ax1.step(sorted_ok["abs_end"], cum, where="post", color="#2980b9", lw=2, label="Cumulative output tokens")
ax1.plot([0, wall], [0, 20_000 * wall / 60], "r--", lw=1.5, label="20k OTPM limit")
ax1.set_ylabel("Cumulative Output Tokens")
ax1.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x/1000:.0f}k"))
ax1.legend(loc="upper left")
ax1.grid(axis="y", alpha=0.3)
ax1.set_title(f"Gemini 3.1 Flash Lite — Throughput vs 20k OTPM\n"
              f"({TOTAL_REQUESTS} reqs, {MAX_WORKERS} workers, {cache_label})", fontweight="bold")

for lbl, sub, c, m, sz in [
    ("OK",       ok[ok["attempts"] == 1], "#2ecc71", "o", 15),
    ("Retried",  ok[ok["attempts"] > 1],  "#f39c12", "^", 30),
    ("Failed",   fail,                     "#e74c3c", "x", 40),
]:
    if not sub.empty:
        ax2.scatter(sub["abs_end"], sub["attempts"] - 1, c=c, marker=m, s=sz, alpha=0.7)

ax2.legend(handles=[
    Line2D([0],[0], marker="o", color="w", markerfacecolor="#2ecc71", ms=6, label="OK"),
    Line2D([0],[0], marker="^", color="w", markerfacecolor="#f39c12", ms=7, label="OK (retried)"),
    Line2D([0],[0], marker="x", color="#e74c3c", ms=7, lw=2, label="Failed"),
], loc="upper right", fontsize=9)
ax2.set(ylabel="Retries", xlabel="Wall-Clock Time (s)")
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

display(df[["req_id","status","attempts","completion_tokens","products_returned","total_elapsed","abs_end","error"]])

### Observations

The chart confirms [burst scaling](https://docs.databricks.com/aws/en/machine-learning/foundation-model-apis/limits#how-limits-are-tracked-and-enforced): the token bucket starts full, so early requests exceed the 20k OTPM line with zero 429s. Once the burst drains, the sliding window throttles and backoff produces the stair-step pattern. Credit-back (~3,500 tokens/req) lets throughput briefly spike above the nominal rate mid-test. All requests eventually succeed.